# Tests: `fastermodels.eval` (source `nbs/01_eval.ipynb`)

In [ ]:
from fastcore.test import *
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from fastermodels.eval import PairedDelta, _mcnemar, agreement, correct_vector, paired_delta, predictions, wilson

In [ ]:
# Wilson against a known table value: 40421 correct out of 50000
test_close(wilson(40421, 50000), (0.80495, 0.81185), eps=1e-4)

# an accuracy of 0 has a lower bound of exactly 0, and the interval stays inside [0, 1]
test_eq(wilson(0, 10)[0], 0.)
assert 0 < wilson(0, 10)[1] < 1
test_eq(wilson(10, 10)[1], 1.)

# a wider z gives a wider interval, more images give a narrower one
_lo95, _hi95 = wilson(90, 100)
_lo99, _hi99 = wilson(90, 100, z=2.576)
assert _lo99 < _lo95 and _hi99 > _hi95
assert (wilson(9000, 10000)[1] - wilson(9000, 10000)[0]) < (_hi95 - _lo95)

with ExceptionExpected(ValueError, regex='n > 0'): wilson(0, 0)

In [ ]:
_a = np.zeros(1000, dtype=bool); _a[:800] = True

# the same model against itself: no difference, no discordant pair
_same = paired_delta(_a, _a)
test_eq(type(_same), PairedDelta)
test_eq(_same.delta, 0.)
assert _same.lo <= 0 <= _same.hi
test_eq(_same.p_mcnemar, 1.)
test_eq(_same.n, 1000)
test_eq(_same.as_dict()['n'], 1000)

# ten more images right out of a thousand: +1.0 point, an interval clear of zero, and a small p
_b = _a.copy(); _b[800:810] = True
_better = paired_delta(_a, _b)
test_close(_better.delta, 1.0, eps=1e-9)
assert _better.lo > 0
assert _better.p_mcnemar < 0.01

# the sign follows b - a
test_close(paired_delta(_b, _a).delta, -1.0, eps=1e-9)

# same seed, same interval
test_eq(paired_delta(_a, _b).as_dict(), _better.as_dict())

# the exact McNemar p only reads the discordant counts
test_eq(_mcnemar(0, 0), 1.0)
test_close(_mcnemar(0, 10), 2 / 2 ** 10, eps=1e-12)
test_eq(_mcnemar(5, 5), 1.0)

with ExceptionExpected(ValueError, regex='same images'): paired_delta(_a, _a[:10])
with ExceptionExpected(ValueError, regex='empty'): paired_delta(np.array([], dtype=bool), np.array([], dtype=bool))

In [ ]:
torch.manual_seed(0)
_X, _y = torch.randn(20, 4), torch.tensor([0, 1] * 10)
_dl = DataLoader(TensorDataset(_X, _y), batch_size=8)
_model = nn.Linear(4, 2).eval()

# the harness agrees with the hand computation, image by image and in dataloader order
_hand = (_model(_X).argmax(1) == _y).numpy()
_cv = correct_vector(_model, _dl)
test_eq(_cv.dtype, np.dtype(bool))
test_eq(_cv.tolist(), _hand.tolist())
test_eq(predictions(_model, _dl).tolist(), _model(_X).argmax(1).tolist())

# a model that is right everywhere and one that is wrong everywhere
test_eq(correct_vector(lambda x: nn.functional.one_hot(torch.tensor([0, 1] * (len(x) // 2)), 2).float(), _dl).all(), True)

# agreement is a fraction over the same images
_p = predictions(_model, _dl)
test_eq(agreement(_p, _p), 1.0)
test_eq(agreement(_p, 1 - _p), 0.0)
test_close(agreement(np.array([0, 1, 2, 3]), np.array([0, 1, 9, 9])), 0.5, eps=1e-12)
with ExceptionExpected(ValueError, regex='same images'): agreement(_p, _p[:2])
with ExceptionExpected(ValueError, regex='empty'): agreement(np.array([]), np.array([]))

# the harness never touches the caller's model: it stays where it was, and in the mode it was in
_where = {n: p.device for n, p in _model.named_parameters()}
correct_vector(_model, _dl, device='cpu')
test_eq({n: p.device for n, p in _model.named_parameters()}, _where)
test_eq(_model.training, False)

In [ ]:
# logits that are not finite raise instead of scoring a silent 0 %
class _Nan(nn.Module):
    def forward(self, x): return torch.full((x.shape[0], 2), float('nan'))

with ExceptionExpected(ValueError, regex='not finite'): correct_vector(_Nan().eval(), _dl)
with ExceptionExpected(ValueError, regex='not finite'): predictions(_Nan().eval(), _dl)

# a model left in training mode raises: its BatchNorm would score the batch, not the model
with ExceptionExpected(ValueError, regex='training mode'): correct_vector(nn.Linear(4, 2).train(), _dl)
with ExceptionExpected(ValueError, regex='training mode'): predictions(nn.Linear(4, 2).train(), _dl)

# an empty dataloader raises
_empty = DataLoader(TensorDataset(torch.zeros(0, 4), torch.zeros(0, dtype=torch.long)), batch_size=8)
with ExceptionExpected(ValueError, regex='empty'): correct_vector(_model, _empty)

# anything callable on a batch works, numpy logits included (an ONNX session is one)
test_eq(predictions(lambda x: np.asarray(_model(x).detach()), _dl).tolist(), _p.tolist())